In [82]:
import pandas as pd
import numpy as np
import torch
from TRGAN.TRGAN_train_load_modules import embeddings, load_model
from Scripts.data_preprocessing_sber import preprocessing_data_from_sber


In [1]:
data = preprocessing_data_from_sber(folder_path=r'Data\Sber\ditry_single')

NameError: name 'preprocessing_data_from_sber' is not defined

In [84]:
onehot_cols = ['PaymentSystem', 'OpType', 'DETAILEDCARDTYPE', 'TRANMETHOD', 'ISOWNTERMINAL', 'NAME', 'MCC', 'DEVICETYPE', 'CurrencyName']
cat_feat_names = ['CustomerKey', 'ID',  'ACCOUNT_ID', 'TERMINAL_CODE', 'CARD',  'TRANS_DETAIL', 'ADDRESS' ]
num_feat_names = ['AMOUNT_EQ', 'HOUR', 'MINUTE', 'SECOND', 'PAY_AMT']
log1p_transform_cols = ['AMOUNT_EQ', 'PAY_AMT']  # если суммы имеют skewed распределение
date_feature = 'DATETIME'
time_feature = ''
client_id = 'ACCOUNT_ID'
mcc_name = 'MCC'
latent_dim = {'onehot': 100, 'categorical': 15, 'numerical': 6, 'cv': 32}

array(['VIRTUAL', 'Санкт-Петербург', 'UNKNOWN', 'Павловск', 'Петергоф',
       'Кронштадт', 'Ломоносов', 'Сестрорецк', 'Пушкин', 'Колпино',
       'Зеленогорск', 'Петродворец', 'Кириши', 'Всеволожск',
       'Красное Село', 'Приозерск', 'Отрадное', 'Шлиссельбург',
       'Сертолово', 'Гатчина', 'Чудово', 'Коммунар', 'Калининград',
       'западнее п. Бугры пересечение КАД и Автодороги Санкт-Петербур- Скотное',
       'Тосно', 'Луга', '199004 Санкт-Петербург'], dtype=object)

In [ ]:
data[data["CITY"]]
data["REGION"].unique()

KeyError: "None of [Index(['VIRTUAL', 'VIRTUAL', 'VIRTUAL', 'VIRTUAL', 'VIRTUAL', 'VIRTUAL',\n       'VIRTUAL', 'VIRTUAL', 'VIRTUAL', 'VIRTUAL',\n       ...\n       'Санкт-Петербург', 'Санкт-Петербург', 'Санкт-Петербург',\n       'Санкт-Петербург', 'Санкт-Петербург', 'Санкт-Петербург', 'Кронштадт',\n       'Санкт-Петербург', 'Санкт-Петербург', 'Санкт-Петербург'],\n      dtype='object', length=275259)] are in the [columns]"

In [85]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 275259 entries, 0 to 275258
Data columns (total 34 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   CustomerKey             275259 non-null  object        
 1   ID                      275259 non-null  object        
 2   AMOUNT_EQ               275259 non-null  float64       
 3   MCC                     275259 non-null  object        
 4   PAY_AMT                 275259 non-null  float64       
 5   CurrencyName            275259 non-null  object        
 6   PaymentSystem           275259 non-null  object        
 7   TERMINAL_CODE           275259 non-null  object        
 8   ADDRESS                 275259 non-null  object        
 9   DEVICETYPE              275259 non-null  object        
 10  TRANS_DETAIL            275259 non-null  object        
 11  NAME                    275259 non-null  object        
 12  OpType                  275259

In [86]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
directory = 'Pretrains_sber_1/'  # папка с сохраненными моделями
experiment_id = 'sber'

In [87]:
len(data['TERMINAL_CODE'].unique())

52299

In [88]:
data[data['TERMINAL_CODE'] == 'NONE']

,CustomerKey,ID,AMOUNT_EQ,MCC,PAY_AMT,CurrencyName,PaymentSystem,TERMINAL_CODE,ADDRESS,DEVICETYPE,...,IS_PHYSICAL_LOCATION,ADDRESS_DETAIL_LEVEL,HAS_PHYSICAL_ADDRESS,IS_VIRTUAL_TRANSACTION,OFFLINE_WITH_ADDRESS,ONLINE_VIRTUAL,DATETIME,HOUR,MINUTE,SECOND


In [89]:
X_emb, X_oh, cond_vector, synth_date, scaler_cat, scaler_onehot, scaler_num, cv_params, scaler, round_array = embeddings(
    data=data,  # можно передать None или заглушку, т.к. load=True
    cat_feat_names=cat_feat_names,
    num_feat_names=num_feat_names,
    onehot_cols=onehot_cols,
    date_feature=date_feature,
    time_feature=time_feature,
    client_id=client_id,
    latent_dim=latent_dim,
    device=device,
    load=True,  # ВАЖНО: загружаем предобученные!
    directory=directory
)

c:\Users\kiril\TRGAN\TRGAN\TRGAN_train_load_modules.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  encoder_onehot.load_state_dict(torch.load(directory + 'onehot_encode

In [90]:
print(round_array)

[2, 0, 0, 0, 2]


In [91]:
scaler_num['index_df_sort']

array([[216979,      0,      0,      0, 216979],
       [ 71335,      1,      1,      1, 161861],
       [108331,      2,      2,      2,  71335],
       ...,
       [ 43312, 274943, 275114, 275122, 176395],
       [223033, 274944, 275221, 275132, 177867],
       [150169, 275142, 275255, 275156, 206171]], shape=(275259, 5))

In [92]:
scaler_num.keys()

dict_keys(['encoder', 'decoder', 'scaler_minmax', 'scaler', 'index_arr', 'index_df_sort'])

In [93]:
scaler_num["decoder"]

Decoder_cont_emb(
  (model): Sequential(
    (0): Linear(in_features=6, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=128, bias=True)
    (5): ReLU()
    (6): Linear(in_features=128, out_features=128, bias=True)
    (7): ReLU()
    (8): Linear(in_features=128, out_features=5, bias=True)
    (9): Tanh()
  )
)

In [94]:
# Загружаем обученные GAN модели
generator, supervisor, loss_array = load_model(
    latent_dim=latent_dim,
    dim_noise=20,  # размерность шума (должен совпадать с обучением)
    experiment_id=experiment_id,
    DIRECTORY=directory,
    DEVICE=device,
)

In [95]:
data_test = data[["ACCOUNT_ID", 'DATETIME']]

In [96]:
from TRGAN.TRGAN_main_V2 import sample, inverse_transform

# Сколько образцов сгенерировать
n_samples = 400000  # столько, сколько нужно

# Генерируем синтетические эмбеддинги и даты
synth_data, synth_date_gen, params = sample(
    n_samples=n_samples,
    generator=generator,
    supervisor=supervisor,
    noise_dim=20,  # размерность шума
    cond_vector=cond_vector,
    X_emb=X_emb,
    encoder=cv_params['encoder'],  # энкдер условного вектора
    data=data,  # нужен для генерации времени (можно передать исходные данные или заглушку)
    date_feature=date_feature,
    name_client_id=client_id,
    time_type='synth',  # или 'initial' если хочешь исходные времена
    cv_params=cv_params,
    device=device
)

In [97]:
synth_date_gen

,DATETIME
0,2017-05-01
1,2017-05-01
2,2017-05-01
3,2017-05-01
4,2017-05-01
...,...
399995,2025-06-05
399996,2025-06-05
399997,2025-06-05
399998,2025-06-05


In [98]:
has_nan = np.isnan(synth_data).any()
print(f"Есть ли NaN в массиве: {has_nan}")

Есть ли NaN в массиве: False


In [99]:
scaler_cat

{'encoder': Encoder_client_emb(
   (model): Sequential(
     (0): Linear(in_features=7, out_features=64, bias=True)
     (1): ReLU()
     (2): Linear(in_features=64, out_features=128, bias=True)
     (3): ReLU()
     (4): Linear(in_features=128, out_features=128, bias=True)
     (5): ReLU()
     (6): Linear(in_features=128, out_features=128, bias=True)
     (7): ReLU()
     (8): Linear(in_features=128, out_features=15, bias=True)
     (9): Tanh()
   )
 ),
 'decoder': Decoder_client_emb(
   (model): Sequential(
     (0): Linear(in_features=15, out_features=64, bias=True)
     (1): ReLU()
     (2): Linear(in_features=64, out_features=128, bias=True)
     (3): ReLU()
     (4): Linear(in_features=128, out_features=128, bias=True)
     (5): ReLU()
     (6): Linear(in_features=128, out_features=128, bias=True)
     (7): ReLU()
     (8): Linear(in_features=128, out_features=7, bias=True)
     (9): Tanh()
   )
 ),
 'scaler': MinMaxScaler(feature_range=(-1, 1)),
 'freq_encoder': [UniformEncoder

In [100]:
scaler_num["index_df_sort"]

array([[216979,      0,      0,      0, 216979],
       [ 71335,      1,      1,      1, 161861],
       [108331,      2,      2,      2,  71335],
       ...,
       [ 43312, 274943, 275114, 275122, 176395],
       [223033, 274944, 275221, 275132, 177867],
       [150169, 275142, 275255, 275156, 206171]], shape=(275259, 5))

In [101]:
X_emb1 = scaler.inverse_transform(X_emb)
synth_data = scaler.inverse_transform(synth_data)
print(synth_data)

synth_df = inverse_transform(synth_data, latent_dim, X_oh.columns, scaler_onehot, scaler_cat, scaler_num, cat_feat_names,
                             mcc_name, num_feat_names, True, synth_date_gen, time_feature, round_array, device=device)

[[-0.42945057 -0.31214878 -0.22568257 ...  0.9140442  -0.6737626
  -0.7764809 ]
 [-0.31244805 -0.12565458 -0.05455771 ...  0.9229527  -0.668902
  -0.7713223 ]
 [-0.9965646   0.99360603 -0.9575724  ...  0.8870467  -0.67399347
  -0.7793547 ]
 ...
 [-0.99866295  0.998934   -0.9981153  ...  0.76841086 -0.3557114
  -0.1760362 ]
 [-0.99832255  0.99219024 -0.99505955 ...  0.72642726 -0.29466197
  -0.05813816]
 [-0.9994533   0.9984907  -0.9812212  ...  0.6880305  -0.3146408
  -0.10699979]]


In [102]:
print(latent_dim['numerical'])

6


In [103]:
synth_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400000 entries, 0 to 399999
Data columns (total 22 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   PaymentSystem     400000 non-null  object        
 1   OpType            400000 non-null  object        
 2   DETAILEDCARDTYPE  400000 non-null  object        
 3   TRANMETHOD        400000 non-null  object        
 4   ISOWNTERMINAL     400000 non-null  object        
 5   NAME              400000 non-null  object        
 6   MCC               400000 non-null  object        
 7   DEVICETYPE        400000 non-null  object        
 8   CurrencyName      400000 non-null  object        
 9   CustomerKey       400000 non-null  object        
 10  ID                400000 non-null  object        
 11  ACCOUNT_ID        400000 non-null  object        
 12  TERMINAL_CODE     400000 non-null  object        
 13  CARD              400000 non-null  object        
 14  TRAN

In [ ]:
synth_df['ISOWNTERMINAL'].unique()

KeyError: 'CITY'

In [105]:
synth_df[synth_df['TERMINAL_CODE'] == 'NONE']

,PaymentSystem,OpType,DETAILEDCARDTYPE,TRANMETHOD,ISOWNTERMINAL,NAME,MCC,DEVICETYPE,CurrencyName,CustomerKey,...,TERMINAL_CODE,CARD,TRANS_DETAIL,ADDRESS,AMOUNT_EQ,HOUR,MINUTE,SECOND,PAY_AMT,DATETIME


In [106]:
print("Ключи в scaler_cat:", scaler_cat.keys())

Ключи в scaler_cat: dict_keys(['encoder', 'decoder', 'scaler', 'freq_encoder', 'index_arr'])


In [107]:
synth_df = synth_df.sample(frac=1, random_state=22)
synth_df.head()

,PaymentSystem,OpType,DETAILEDCARDTYPE,TRANMETHOD,ISOWNTERMINAL,NAME,MCC,DEVICETYPE,CurrencyName,CustomerKey,...,TERMINAL_CODE,CARD,TRANS_DETAIL,ADDRESS,AMOUNT_EQ,HOUR,MINUTE,SECOND,PAY_AMT,DATETIME
386986,MasterCard,Оплата,MasterCard World,70,0,Оплата товаров/услуг по карте,5331,VIRTUAL,Рубль,1010550,...,26026514,2109285,"KARUSEL, SANKT-PETERBU, RU",VIRTUAL_TRANSACTION,209.74,13,15,15,204.71,2017-05-01 17:21:39
398961,MasterCard,Оплата,MasterCard Unembossed,51,0,Оплата товаров/услуг по карте,5992,VIRTUAL,Рубль,1007032,...,00048217,259605,"PYATEROCHKA 428, LOMONOSOV, RU",VIRTUAL_TRANSACTION,121.62,11,7,7,119.50,2017-05-13 17:17:33
314025,MasterCard,Оплата,MasterCard Unembossed,51,0,Оплата товаров/услуг по карте,5999,VIRTUAL,Рубль,1007896,...,M5500022,2567987,"PRISMA, SANKT-PETERBU, RU",VIRTUAL_TRANSACTION,151.14,12,10,10,148.20,2017-11-30 00:00:00
386195,MasterCard,Снятие наличных,MasterCard World,51,1,Выдача наличных ден. средств через банкомат,6011,ATM,Рубль,1010674,...,441924,2580187,"WHSD SOUTH, SANKT-PETERBU, RU",VIRTUAL_TRANSACTION,174.49,12,12,12,170.73,2018-01-02 13:32:34
322196,MasterCard,Оплата,MasterCard Standard,70,0,Оплата товаров/услуг по карте,5462,VIRTUAL,Рубль,1006683,...,297108,2490975,"GLORIA JEANS, KOLPINO, RU",VIRTUAL_TRANSACTION,952.92,18,43,43,937.63,2018-05-28 14:21:48


In [108]:
synth_df.to_csv('Data/Sber/Clear/transact_400000_samples.csv', 
          index=False,           # Не записывать индексы
          sep=',',               # Разделитель
          encoding='utf-8',      # Кодировка
          header=True,           # Записывать заголовки
          na_rep='NULL')         # Замена NaN значений